# Lab 8 · Chuỗi thời gian: quý, cửa sổ trượt & so cùng kỳ

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 8**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 8 làm việc theo **tháng**. Lab này đổi độ phân giải: xuống tận
**ngày** (cửa sổ trượt 7 ngày) và lên tận **quý** — rồi tự tay tính một con số
"so cùng kỳ" đúng chuẩn báo cáo.

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🔒 ở giờ lý thuyết đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn 🔓: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Cắt lát thời gian bằng chuỗi (`.loc["2019"]`) và đối chiếu với con số đã biết.
2. `resample` theo quý; nhận diện và loại kỳ cụt ở cấp quý.
3. Làm mượt chuỗi ngày bằng `rolling` và đọc đỉnh/đáy.
4. Tính "so cùng kỳ" (YoY) và phân biệt nó với "so kỳ liền trước".

## Phần 0 · Khởi động (~8 phút)

In [ ]:
import pandas as pd

# W1 — nạp reviews, parse ngày, đưa date vào index và sắp xếp
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/reviews.csv")
rv = pd.read_csv(URL, parse_dates=["date"])
# TODO: tạo r = rv với index là date, đã sort_index
r = ...

# --- Ô kiểm tra ---
assert str(type(r.index)).endswith("DatetimeIndex'>")
assert r.index.is_monotonic_increasing
print("W1 ổn — trục thời gian sẵn sàng.")

In [ ]:
# W2 — cắt lát bằng chuỗi + đối chiếu số đã biết
# Ở lab 2 bạn đã đếm bằng dict: 2019 có 24.299 review, 2021 có 23.926.
# TODO: đếm lại bằng cắt lát chuỗi trên r
n_2019 = ...
n_2021 = ...

# --- Ô kiểm tra ---
assert n_2019 == 24299 and n_2021 == 23926
print(f"Khớp lab 2. Đến 2021 thị trường đã hồi {n_2021 / n_2019:.0%} mức 2019.")

## Phần 1 · Nhìn theo quý (~25 phút)

### Bước 1 · resample quý và kỳ cụt cấp quý

In [ ]:
# TODO: đếm review theo QUÝ (mã tần suất "QE")
quy = ...

# --- Ô kiểm tra ---
assert quy.loc["2026-03-31"] == 68395     # quý 1/2026
quy.tail(4)

Nhìn 4 quý cuối: Q3/2026 chỉ có **49** review — snapshot chụp 29/06 nên "quý 3" mới
có vài giờ dữ liệu. Và để ý tinh vi hơn: **Q2/2026 cũng hụt** (thiếu ngày 30/06)!
Kỳ cụt không phải lúc nào cũng lộ liễu như con số 49 — kỳ "gần trọn" mới nguy hiểm,
vì nhìn qua tưởng bình thường.

In [ ]:
# TODO: tạo quy_du — bỏ MỌI quý chưa trọn (2 quý cuối), rồi tìm quý đỉnh & quý đáy
#        trong giai đoạn 2019–2021 (dùng .loc cắt lát trước khi idxmin)
quy_du = ...
quy_dinh = ...          # nhãn thời gian của quý cao nhất toàn lịch sử (idxmax)
quy_day_covid = ...     # nhãn của quý THẤP nhất trong 2019–2021 (idxmin trên lát cắt)

# --- Ô kiểm tra ---
assert len(quy_du) == len(quy) - 2
assert str(quy_dinh)[:10] == "2026-03-31"
assert str(quy_day_covid)[:10] == "2020-06-30" and quy.loc[quy_day_covid] == 1022
print(f"Đỉnh: Q1/2026 ({quy_du.max():,}) — Đáy COVID: Q2/2020 ({quy.loc[quy_day_covid]:,}).")

Q2/2020 — quý phong toả đầu tiên — chỉ 1.022 review, bằng 1/67 quý đỉnh.
Hai con số này (đỉnh, đáy, kèm mốc) là loại "số biết nói" nên có trong báo cáo.

## Phần 2 · Xuống độ phân giải ngày + rolling (~22 phút)

In [ ]:
# TODO: đếm review theo NGÀY của riêng năm 2025 (cắt lát trước, resample "D" sau)
ngay_2025 = ...

# TODO: làm mượt bằng cửa sổ trượt 7 ngày, căn giữa (rolling(7, center=True).mean())
tb7 = ...

# --- Ô kiểm tra ---
assert len(ngay_2025) == 365
assert ngay_2025.max() == 1460
assert str(tb7.idxmax())[:10] == "2025-11-23"
print(f"Ngày bận nhất 2025 (theo trung bình 7 ngày): {tb7.idxmax():%d/%m/%Y} — {tb7.max():.0f} review/ngày.")

In [ ]:
# Vẽ nhanh để thấy rolling làm gì (code cho sẵn — buổi 12 học kỹ)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.plot(ngay_2025.index, ngay_2025.values, color="#bbb", lw=0.8, label="theo ngày")
ax.plot(tb7.index, tb7.values, color="#1E93AB", lw=2, label="trượt 7 ngày")
ax.legend(frameon=False)
ax.set_title("Nhịp review 2025: răng cưa tuần biến mất khi làm mượt 7 ngày")
plt.show()

Đường xám răng cưa theo **tuần** (khách trả phòng cuối tuần); cửa sổ 7 ngày nuốt
đúng chu kỳ đó nên đường xanh chỉ còn xu hướng + mùa vụ. Chọn **độ rộng cửa sổ bằng chu kỳ
nhiễu** là một quyết định có lý do — không phải con số tuỳ hứng.

## Phần 3 · So cùng kỳ — con số báo cáo được (~15 phút)

Tháng 5/2026 có 22.296 review. Nhiều hay ít? Câu trả lời phụ thuộc **so với gì**.

In [ ]:
thang = r.resample("ME").size()

# TODO: tính 2 phép so cho tháng 5/2026:
so_ky_truoc = ...    # % thay đổi so với THÁNG 4/2026 (dùng .loc hai nhãn, tự tính %)
so_cung_ky = ...     # % thay đổi so với THÁNG 5/2025

# --- Ô kiểm tra ---
assert round(so_ky_truoc, 1) == 3.0
assert round(so_cung_ky, 1) == 53.7
print(f"T5/2026: +{so_ky_truoc:.0f}% so tháng trước · +{so_cung_ky:.0f}% so cùng kỳ.")

Hai con số kể hai chuyện khác nhau: "+3% so tháng trước" (nhịp ngắn hạn, dính mùa vụ)
và "+54% so cùng kỳ" (đà tăng thật của thị trường). Báo cáo bài tập lớn: dữ liệu có mùa vụ
thì **so cùng kỳ là phép so mặc định**, so kỳ trước chỉ để mô tả nhịp.

## Phần 4 · Bài tự làm 🔓 (làm xong sớm / về nhà)

Được dùng AI theo quy trình 5 bước; ghi lại prompt + cách kiểm chứng.

### Tự làm 1 · Điều tra ngày kỷ lục

Tìm **ngày có nhiều review nhất toàn lịch sử** (resample "D" trên cả chuỗi). Bạn sẽ thấy
một ngày giữa tháng 3/2026 với hơn 2.000 review — cao gấp rưỡi mọi ngày quanh nó, và là
**thứ Hai**. Hãy: (1) in số review của 5 ngày quanh nó; (2) tra thử tuần đó ở Santiago có
sự kiện gì (gợi ý: một lễ hội âm nhạc lớn thường diễn ra giữa tháng 3); (3) viết 2 câu
kết luận kiểu báo cáo — có dè chừng đúng mực (đây là *một* cách giải thích, chưa kiểm chứng
độc lập).

### Tự làm 2 · "Tuổi" listing bằng Timedelta

Với bảng listings (bản rút gọn): tính số ngày từ `last_review` đến ngày snapshot 29/06/2026.
Bao nhiêu listing có review gần nhất **cách đây hơn 1 năm** (>365 ngày)? Nhóm này nói lên
điều gì về "listing zombie" — và nên gắn cờ gì cho pipeline bài tập lớn?

In [ ]:
# Viết bài tự làm của bạn ở đây

## Tóm tắt buổi lab

| Bạn đã làm | Sẽ gặp lại ở |
|---|---|
| Cắt lát chuỗi + đối chiếu số lab 2 | kiểm chứng chéo giữa các công cụ |
| Kỳ cụt cấp quý — kể cả kỳ "gần trọn" | mọi biểu đồ thời gian của bài tập lớn |
| rolling 7 ngày nuốt chu kỳ tuần | trực quan hoá chuỗi thời gian (buổi 12) |
| So kỳ trước vs so cùng kỳ | KPI thời gian chuẩn báo cáo |

Buổi lý thuyết tới (sau giữa kỳ): **làm sạch dữ liệu** — nơi mọi nghi vấn bạn đã gắn cờ
từ lab 2 đến giờ được xử lý có hệ thống. Tuần sau: **thi giữa kỳ** — ôn theo deck buổi 9.